## Задание 1

```
корпус:
1) Погода сегодня отличная! => погода сегодня отличная
2) Отличная температура, но погода дождливая :( => отличная температура но погода дождливая
3) Сегодня дождь, дождь - грустно => сегодня дождь дождь грустно

уникальные слова: грустно, дождливая, дождь, но, отличная, погода, сегодня, температура
```

```
грустно	дождливая	дождь	но	отличная	погода	сегодня	температура
№1	0	0	0	0	1	1	1	0
№2	0	1	0	1	1	1	0	1
№3	1	0	2	0	0	0	1	0
```

```
TF:
    1) (6 слов): отличная(1/6), погода(1/6), сегодня(1/6)
    2) (6 слов): отличная(1/6), температура(1/6), но(1/6), погода(1/6), дождливая(1/6)
    3) (4 слова): сегодня(1/4), дождь(2/4=0.5), грустно(1/4)
```

```
IDF:

слово	встречается в документах	IDF = log(3 / кол-во док-в)
грустно	       1	log(3) = 1.099
дождливая	   1	1.099
дождь	       1	1.099
но	           1	1.099
отличная	   2	log(1.5) = 0.405
погода	       2	0.405
сегодня	       2	0.405
температура	   1	1.099
```

```
TF-IDF:

   грустно	дождлив	дождь	но	отличная	погода	сегодня	темпер
№1	0	0	0	0	0.068	0.068	0.068	0
№2	0	0.183	0	0.183	0.068	0.068	0	0.183
№3	0.275	0	0.55	0	0	0	0.101	0
```

```
Ответ:
    1) отличная, погода, сегодня
    2) дождливая, но, температура
    3) дождь (самое важное), грустно

    контекстное слово: дождь / погода
```

## Задание 2

In [1]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

data = [
    "Погода сегодня отличная",
    "Отличная температура но погода дождливая",
    "Сегодня дождь дождь грустно"
]

data = [s.lower() for s in data]
data = [''.join(filter(lambda s: s.isalpha() or s.isspace(), s)) for s in data]

Bag_of_Words = CountVectorizer()
New_data = Bag_of_Words.fit_transform(data)
New_data = New_data.toarray()
print(New_data)
print(sorted(Bag_of_Words.vocabulary_.keys()))

tf_idf = TfidfVectorizer()
New_data = tf_idf.fit_transform(data)
feature_names = tf_idf.get_feature_names_out()

df = pd.DataFrame(New_data.toarray().round(3),
                  columns=feature_names,
                  index=["D1", "D2", "D3"])
print(df)

[[0 0 0 0 1 1 1 0]
 [0 1 0 1 1 1 0 1]
 [1 0 2 0 0 0 1 0]]
['грустно', 'дождливая', 'дождь', 'но', 'отличная', 'погода', 'сегодня', 'температура']
    грустно  дождливая  дождь    но  отличная  погода  сегодня  температура
D1    0.000       0.00  0.000  0.00     0.577   0.577    0.577         0.00
D2    0.000       0.49  0.000  0.49     0.373   0.373    0.000         0.49
D3    0.423       0.00  0.847  0.00     0.000   0.000    0.322         0.00


## Задание 3

```
Корпус:
Клара украла кларнет
К_ад_кр_л пи_ат

Решение
после очистки: клара украла кларнет
кл, ла, ар, ра, у, ук, кр, ра, а, кл, ла, ар, рн, не, ет

Частоты:
    "кл" = 2 (клара, кларнет)
    "ла" = 2
    "ар" = 2
    "ра" = 2
    "ук" = 1
    "кр" = 1
    "рн" = 1
    "не" = 1
    "ет" = 1

Вероятность P(буква2 | буква1) = частота(буква1+буква2) / частота(буква1)

К_ад_кр_л пи_ат вероятные буквы по контексту:
    К(а?)ад => после "К" часто идёт "л" (клара, кларнет) => Кл
    ад_(к?) => после пробел перед "к" - нет данных, но "кр" есть => Клад
    кр_л => между "кр" и "л" возможна "а" (как в клара) => крал
    пи_ат → между "пи" и "ат": "с" (писат) - "писать".

Ответ: Клад крал писал
```

## Задание 3

In [2]:
import nltk
from nltk import bigrams
from collections import Counter
import re

text = '''Он зажег свечу и осмотрел нумер подробнее Это была клетушка до того маленькая что даже почти не под рост Свидригайлову в одно окно постель очень грязная простой крашеный стол и стул занимали почти все пространство Стены имели вид как бы сколоченных из досок с обшарканными обоями до того уже пыльными и изодранными что цвет их желтый угадать еще можно было но рисунка уже нельзя было распознать никакого Одна часть стены и потолка была срезана накось как обыкновенно в мансардах но тут над этим косяком шла лестница'''

text = re.sub(r'[^\w\s]', '', text.lower())
text = text.replace('ё', 'е')
text = text.replace(' ', '')
chars = list(text)

bigram_list = list(bigrams(chars))
bigram_freq = Counter(bigram_list)
unigram_freq = Counter(chars)

cond_prob = {}
for (c1, c2), cnt in bigram_freq.items():
    cond_prob[(c1, c2)] = cnt / unigram_freq[c1]

prob = sorted(cond_prob.items(), key=lambda x: -x[1])[:5]
count = sorted(bigram_freq.items(), key=lambda x: -x[1])[:5]

print('по вероятности:')
print(*[i for i in prob], sep='\n')
print('по частоте:')
print(*[i for i in count], sep='\n')

second = "н_ ст_ле догор_ла св_ча"
fill = {
    "н_": "а",
    "ст_ле": "о",  # столе
    "догор_ла": "е",  # догорела
    "св_ча": "е"  # свеча
}

по вероятности:
(('э', 'т'), 1.0)
(('щ', 'е'), 1.0)
(('ж', 'е'), 0.8333333333333334)
(('п', 'о'), 0.7)
(('б', 'ы'), 0.6666666666666666)
по частоте:
(('с', 'т'), 11)
(('о', 'с'), 9)
(('т', 'о'), 8)
(('к', 'а'), 8)
(('е', 'н'), 8)


## Задание 4

```
Слово	Всего писем (20)	Спам (12)	Не спам (8)
акция	6	5	1
выигрыш	5	4	1
срочно	4	4	0
бесплатно	7	6	1
деньги	3	2	1
```

```
Pr(S) = 12/20 = 0.6, Pr(H) = 0.4

акция: (5/12 * 0.6) / ( (5/12*0.6) + (1/8*0.4) ) = 0.25 / (0.25 + 0.05) = 0.833
выигрыш: (4/12*0.6)/((4/12*0.6)+(1/8*0.4)) = 0.2/(0.2+0.05)=0.8
срочно: (4/12*0.6)/((4/12*0.6)+(0)*0.4)=1.0
бесплатно: (6/12*0.6)/((6/12*0.6)+(1/8*0.4))=0.3/(0.3+0.05)=0.857
деньги: (2/12*0.6)/((2/12*0.6)+(1/8*0.4))=0.1/(0.1+0.05)=0.667

итог:
p = (0.833*0.8*1*0.857*0.667) / (0.833*0.8*1*0.857*0.667 + (0.167*0.2*0*0.143*0.333))

знаменатель обнуляется из-за срочно => 100% спам
```

## Задание 5

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('spam(1).csv', encoding='ISO-8859-1')
df = df.drop(columns=['v3', 'v4', 'v5'])
df.head()


,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
f = df[['v1', 'v2']]
df.columns = ['label', 'message']
df['label_enc'] = df['label'].map({'spam': 1, 'ham': 0})

X_train, X_test, y_train, y_test = train_test_split(df['message'], df['label_enc'], test_size=0.2, random_state=42)

vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
print(f"Точность модели: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

new_messages = [
    "Congratulations!!! You won a $1000 gift card. Click here to claim now.",
    "Hey, are we still meeting for lunch tomorrow?"
]
new_messages_vec = vectorizer.transform(new_messages)
predictions = model.predict(new_messages_vec)

for msg, pred in zip(new_messages, predictions):
    print(f"Сообщение: '{msg}'\nПредсказание: {'spam' if pred == 1 else 'ham'}\n")

Точность модели: 0.9830
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       965
           1       0.99      0.89      0.93       150

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115

Сообщение: 'Congratulations!!! You won a $1000 gift card. Click here to claim now.'
Предсказание: spam

Сообщение: 'Hey, are we still meeting for lunch tomorrow?'
Предсказание: ham

